# HSQM 국소 역확산 안정화판 — 오류 삭제와 정상 획 보호

L4에서 1 → 2 → 3 → 4 실행. 기본 설정을 고정한 단일 실행 흐름입니다.
입력: 원래 sweep ZIP 또는 획 묶음이 포함된 adaptive_ink_reverse 결과 ZIP.
2번에서 기존 획 추출 모델을 Drive에서 찾아 SHA256을 검증합니다. 직접 업로드도 가능합니다.
최신 adaptive ZIP에서도 보정 전 원래 입력부터 비교하며, 손상된 이전 출력을 이어 쓰지 않습니다.

성공했던 고정 마스크를 기본으로 쓰고, 재추출 제안은 2회 연속 확인된 경우만 반영합니다.
획별 정상 영역에 대해 몸통 평균 오차, 국소 오차, 잉크 감소 한도를 검사합니다.
한도를 넘는 latent 수정은 거절합니다. 전체 출력도 재검사하여 실패하면 원래 입력을 보존합니다.
최종 선택본은 selected.png, 원시 역확산 결과와 거절 기록도 모두 저장합니다.
이는 내부 보호 조건이며 시각적 정상성/완전 복원의 보증이 아닙니다.
candidate RGB를 복제하거나 결과 픽셀을 지우는 후처리는 하지 않습니다.

In [ ]:
import sys,subprocess,json,hashlib,io,zipfile
from pathlib import Path
subprocess.run([sys.executable,'-m','pip','install','diffusers==0.37.0',
    'transformers==4.57.6','accelerate','safetensors','timm','numpy','pillow',
    'matplotlib','scipy','scikit-image'],check=True)
RUNTIME_HELPERS={'diffusion_model/fitted_candidate_diffusion.py': '"""Paired DDIM re-denoising guided by an extractor-fitted candidate, not GT."""\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch.nn import functional as F\n\n\ndef load_case(archive, case_id):\n    """Read named members only; recorded Colab absolute paths are not trusted."""\n    with zipfile.ZipFile(archive) as z:\n        if \'case_manifest.json\' not in z.namelist():\n            report = json.loads(z.read(\'report.json\'))\n            metadata = dict(report[\'metadata\'])\n            if metadata[\'case\'][\'case_id\'] != case_id:\n                raise ValueError(\'CASE_ID does not match the saved reverse experiment\')\n            with Image.open(io.BytesIO(z.read(\'input.png\'))) as im:\n                source = im.convert(\'RGB\').copy()\n            with Image.open(io.BytesIO(z.read(\'fitted_candidate.png\'))) as im:\n                target = im.convert(\'RGB\').copy()\n            if source.size != target.size or any(n % 8 for n in source.size):\n                raise ValueError(\'Input/candidate dimensions mismatch\')\n            metadata[\'reused_archive_kind\'] = \'reverse_result_original_input_and_candidate\'\n            return source, target, metadata\n        cases = json.loads(z.read(\'case_manifest.json\'))\n        matches = [c for c in cases if c[\'case_id\'] == case_id]\n        if len(matches) != 1:\n            raise ValueError(\'Select exactly one case from case_manifest.json\')\n        case = matches[0]\n        page = f"generation/page_{case[\'page_id\']:03d}.png"\n        candidate = f"{case_id}/normal_candidate.png"\n        def read_image(name):\n            with Image.open(io.BytesIO(z.read(name))) as im:\n                return im.convert(\'RGB\').copy()\n        source, target = read_image(page), read_image(candidate)\n        if source.size != target.size or any(n % 8 for n in source.size):\n            raise ValueError(\'Input and fitted candidate must have equal dimensions divisible by 8\')\n        report = json.loads(z.read(\'generation/generation_report.json\'))\n        metadata = dict(case=case, candidate_member=candidate, input_member=page,\n                        generation_model=report[\'model_id\'],\n                        candidate_is_ground_truth=False,\n                        prompt=report.get(\'prompts\', {}).get(\'4\', \'\'),\n                        negative_prompt=report.get(\'negative_prompt\', \'\'))\n    return source, target, metadata\n\n\ndef image_tensor(image, device):\n    a = np.asarray(image, dtype=np.float32) / 255\n    return torch.from_numpy(a.copy()).permute(2, 0, 1)[None].to(device)\n\n\nclass InkConstraintObjective:\n    """Candidate supplies fixed sparse constraints, never RGB target pixels."""\n    def __init__(self, source, candidate, *, enable_restore=False, tolerance=5.,\n                 halo=2, preserve_weight=.5, background_ink=.015, required_ink=.7):\n        from scipy.ndimage import distance_transform_edt, binary_dilation\n        from skimage.morphology import skeletonize\n        if source.shape != candidate.shape or source.shape[:2] != (1, 3):\n            raise ValueError(\'Expected matching single RGB sheets\')\n        if not np.isfinite(tolerance) or tolerance < 2 or type(halo) is not int or halo < 0:\n            raise ValueError(\'Invalid constraint tolerance/halo\')\n        if not 0 <= background_ink < required_ink <= 1 or preserve_weight < 0:\n            raise ValueError(\'Invalid ink constraints\')\n        self.source = source.detach()\n        self.preserve_weight = preserve_weight\n        self.background_ink, self.required_ink = background_ink, required_ink\n        source_ink = self.ink(source).detach().cpu().numpy()[0, 0]\n        target_ink = self.ink(candidate).detach().cpu().numpy()[0, 0]\n        source_binary, target_binary = source_ink > .35, target_ink > .35\n        if not source_binary.any() or not target_binary.any():\n            raise ValueError(\'Empty source or candidate: cannot infer safe correspondence\')\n        distance = distance_transform_edt(~target_binary)\n        # Ignore ordinary boundary/width differences within a tolerance corridor.\n        seeds = source_binary & (distance > tolerance)\n        nearby = binary_dilation(seeds, iterations=halo) if halo else seeds\n        erase = nearby*np.clip((distance-1)/max(tolerance-1, 1), 0, 1)\n        # Restore only an absent centerline, not the candidate\'s full silhouette.\n        missing = skeletonize(target_binary) & (distance_transform_edt(~source_binary) > tolerance)\n        restore = missing.astype(np.float32) if enable_restore else np.zeros_like(erase)\n        erase[restore > 0] = 0  # Never issue conflicting instructions.\n        def tensor(a):\n            return torch.as_tensor(a.copy(), dtype=torch.float32, device=source.device)[None, None]\n        self.erase, self.restore = tensor(erase), tensor(restore)\n        self.edit = torch.maximum(self.erase, self.restore)\n        occupied = (self.edit > 0).float()\n        self.preserve = 1-F.max_pool2d(occupied, 5, 1, 2)\n        self.metadata = dict(enable_restore=enable_restore, tolerance_pixels=tolerance,\n            halo_pixels=halo, preserve_weight=preserve_weight,\n            background_ink=background_ink, required_ink=required_ink,\n            erase_pixels=int((self.erase > 0).sum()), restore_pixels=int((self.restore > 0).sum()),\n            candidate_rgb_matching=False, masks_frozen_for_entire_pass=True)\n\n    @staticmethod\n    def ink(rgb):\n        return 1-(rgb*rgb.new_tensor([.299, .587, .114])[None, :, None, None]).sum(1, keepdim=True)\n\n    def project_gradient(self, gradient):\n        # A small latent context halo permits VAE reconstruction at edit boundaries.\n        support = F.adaptive_max_pool2d((self.edit > 0).float(), gradient.shape[-2:])\n        return gradient*F.max_pool2d(support, 3, 1, 1)\n\n    def __call__(self, image):\n        ink = self.ink(image)\n        remaining = F.relu(ink-self.background_ink)\n        erase = CandidateObjective.mean_region(remaining, self.erase)\n        # Penalize the darkest residual pixels as well as average remaining ink.\n        active = remaining[self.erase > 0]\n        tail = active.topk(max(1, int(active.numel()*.1))).values.mean() if active.numel() else erase*0\n        restore = CandidateObjective.mean_region(F.relu(self.required_ink-ink), self.restore)\n        preserve = CandidateObjective.mean_region((image-self.source).square(), self.preserve)\n        healthy_ink = self.preserve*(self.ink(self.source) > .2).float()\n        healthy_mse = CandidateObjective.mean_region((image-self.source).square(), healthy_ink)\n        loss = erase+.25*tail+restore+self.preserve_weight*preserve\n        return loss, dict(erase_ink=float(erase.detach()), erase_tail=float(tail.detach()),\n            restore_deficit=float(restore.detach()), preserved_region_mse=float(preserve.detach()),\n            healthy_ink_rgb_mse=float(healthy_mse.detach()))\n\n\nclass CandidateObjective:\n    """Fixed spatial weights; gradients stay continuous through the VAE."""\n    def __init__(self, source, candidate, threshold=.10, radius=2, preserve_weight=.25):\n        if source.shape != candidate.shape or not 0 < threshold < 1 or radius < 0:\n            raise ValueError(\'Invalid images or edit-region settings\')\n        self.source, self.candidate = source.detach(), candidate.detach()\n        difference = (source - candidate).abs().mean(1, keepdim=True)\n        edit = (difference > threshold).float()\n        self.edit = F.max_pool2d(edit, 2*radius+1, 1, radius).detach()\n        self.preserve_weight = preserve_weight\n\n    @staticmethod\n    def mean_region(values, mask):\n        return (values*mask).sum() / (mask.sum()*values.shape[1]).clamp_min(1)\n\n    def __call__(self, image):\n        match = self.mean_region((image-self.candidate).square(), self.edit)\n        preserve = self.mean_region((image-self.source).square(), 1-self.edit)\n        loss = match + self.preserve_weight*preserve\n        return loss, dict(candidate_region_mse=float(match.detach()),\n                          preserved_region_mse=float(preserve.detach()))\n\n\ndef optimize_clean(clean, decode, objective, *, iterations=4, step_ratio=.02,\n                   budget_ratio=.08, backtracks=5):\n    """Backtrack on the SAME clean estimate; bound total correction per step."""\n    anchor = clean.detach().float()\n    current = anchor.clone()\n    scale = anchor.square().mean().sqrt().clamp_min(1e-3)\n    with torch.no_grad():\n        before, _ = objective(decode(current))\n    accepted = 0\n    rejected_guards = 0\n    for _ in range(iterations):\n        with torch.enable_grad():\n            leaf = current.detach().requires_grad_(True)\n            loss, _ = objective(decode(leaf))\n            gradient, = torch.autograd.grad(loss, leaf)\n            if hasattr(objective, \'project_gradient\'):\n                gradient = objective.project_gradient(gradient)\n        if not torch.isfinite(gradient).all() or not torch.isfinite(loss):\n            raise FloatingPointError(\'Nonfinite candidate guidance\')\n        norm = gradient.square().mean().sqrt()\n        if norm < 1e-12:\n            break\n        direction = step_ratio*scale*gradient/norm\n        found = False\n        with torch.no_grad():\n            before_image = decode(current) if hasattr(objective, \'check_proposal\') else None\n            for attempt in range(backtracks+1):\n                proposal = current-direction*(.5**attempt)\n                delta = proposal-anchor\n                rms = delta.square().mean().sqrt().clamp_min(1e-12)\n                proposal = anchor+delta*(budget_ratio*scale/rms).clamp(max=1)\n                trial_image = decode(proposal)\n                trial, _ = objective(trial_image)\n                guard_ok = True\n                if torch.isfinite(trial) and trial < loss.detach() and hasattr(objective, \'check_proposal\'):\n                    guard_ok = objective.check_proposal(before_image, trial_image)\n                    rejected_guards += int(not guard_ok)\n                if torch.isfinite(trial) and trial < loss.detach() and guard_ok:\n                    current = proposal.detach()\n                    accepted += 1\n                    found = True\n                    break\n        if not found:\n            break\n    with torch.no_grad():\n        after, details = objective(decode(current))\n    return current, dict(loss_before=float(before), loss_after=float(after),\n                         accepted_inner_steps=accepted, rejected_guard_attempts=rejected_guards, **details)\n\n\ndef run_pair(source, candidate, metadata, output, *, steps=60, strength=.35,\n             seed=2026, cfg_scale=5., inner_steps=4, step_ratio=.02,\n             model_id=\'stable-diffusion-v1-5/stable-diffusion-v1-5\', revision=None,\n             progress=None, guidance_mode=\'candidate\', constraint_config=None, objective_override=None):\n    """Identical source VAE encoding, noise, prompt, scheduler in both arms."""\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Select an L4 GPU runtime in Colab\')\n    if not 0 < strength <= 1 or steps < 4 or int(steps*strength) < 2:\n        raise ValueError(\'Need at least two actual re-denoising steps\')\n    if inner_steps < 1 or step_ratio <= 0 or cfg_scale <= 1:\n        raise ValueError(\'Invalid guidance configuration\')\n    if guidance_mode not in (\'candidate\', \'ink_constraints\', \'adaptive_ink\'):\n        raise ValueError(\'Unknown guidance mode\')\n    if (guidance_mode == \'adaptive_ink\') != (objective_override is not None):\n        raise ValueError(\'Adaptive mode requires an explicitly prepared stroke objective\')\n    from diffusers import StableDiffusionImg2ImgPipeline, DDIMScheduler\n    from copy import deepcopy\n    out = Path(output)\n    out.mkdir(parents=True, exist_ok=False)\n    source.save(out/\'input.png\')\n    candidate.save(out/\'fitted_candidate.png\')\n    kwargs = dict(torch_dtype=torch.float16, use_safetensors=True)\n    if revision:\n        kwargs[\'revision\'] = revision\n    pipe = StableDiffusionImg2ImgPipeline.from_pretrained(model_id, **kwargs)\n    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config,\n                                              clip_sample=False, thresholding=False)\n    pipe.to(\'cuda\')\n    pipe.enable_attention_slicing()\n    pipe.enable_vae_slicing()\n    for module in (pipe.unet, pipe.vae, pipe.text_encoder):\n        module.eval().requires_grad_(False)\n    prompt = metadata.get(\'prompt\') or (\'four Korean Hangul glyphs, black ink on white, \'\n                                      \'preserve the input typography and layout\')\n    with torch.no_grad():\n        positive, negative = pipe.encode_prompt(prompt, \'cuda\', 1, True,\n            negative_prompt=metadata.get(\'negative_prompt\', \'\'))\n        embeddings = torch.cat([negative, positive])\n        pipe.scheduler.set_timesteps(steps, device=\'cuda\')\n        timesteps, _ = pipe.get_timesteps(steps, strength, \'cuda\')\n        # VAE and noise initialization exactly once, then copy for both arms.\n        initial = pipe.prepare_latents(pipe.image_processor.preprocess(source), timesteps[:1],\n            1, 1, embeddings.dtype, \'cuda\', torch.Generator(\'cpu\').manual_seed(seed))\n    pipe.vae.to(dtype=torch.float32)\n    def decode(z):\n        return pipe.vae.decode(z.float()/pipe.vae.config.scaling_factor,\n                               return_dict=False)[0].float()/2+.5\n    def save_image(z, path):\n        with torch.no_grad():\n            array = decode(z)[0].permute(1, 2, 0).clamp(0, 1).cpu().numpy()\n        Image.fromarray(np.rint(array*255).astype(\'uint8\')).save(path)\n    source_rgb, candidate_rgb = image_tensor(source, \'cuda\'), image_tensor(candidate, \'cuda\')\n    if objective_override is not None and (objective_override.source.shape != source_rgb.shape or\n            not torch.allclose(objective_override.source, source_rgb, atol=1e-6)):\n        raise ValueError(\'Adaptive objective must be prepared from this exact original input\')\n    objective = objective_override if objective_override is not None else (InkConstraintObjective(source_rgb, candidate_rgb, **(constraint_config or {}))\n                 if guidance_mode == \'ink_constraints\' else CandidateObjective(source_rgb, candidate_rgb))\n    Image.fromarray((objective.edit[0, 0].cpu().numpy()*255).astype(\'uint8\')).save(out/\'edit_region.png\')\n    if guidance_mode in (\'ink_constraints\', \'adaptive_ink\'):\n        for name in (\'erase\', \'restore\', \'preserve\'):\n            Image.fromarray(np.rint(getattr(objective, name)[0, 0].cpu().numpy()*255).astype(\'uint8\')).save(out/f\'{name}_region.png\')\n        (out/\'constraints.json\').write_text(json.dumps(objective.metadata, indent=2), encoding=\'utf-8\')\n    guided_name = {\'candidate\': \'candidate_guided\', \'ink_constraints\': \'ink_guided\', \'adaptive_ink\': \'adaptive_guided\'}[guidance_mode]\n    if hasattr(objective, \'export_state\'):\n        objective.export_state(out/\'ownership_initial.npz\')\n    traces, metrics = {}, {}\n    for method in (\'redenoise_only\', guided_name):\n        scheduler = deepcopy(pipe.scheduler)\n        latent = initial.detach().float().clone()\n        trace = []\n        for i, t in enumerate(timesteps):\n            with torch.no_grad():\n                inputs = scheduler.scale_model_input(torch.cat([latent, latent]), t)\n                u, c = pipe.unet(inputs.to(torch.float16), t,\n                    encoder_hidden_states=embeddings, return_dict=False)[0].chunk(2)\n                noise = (u+cfg_scale*(c-u)).float()\n                base = scheduler.step(noise, t, latent, eta=0., return_dict=True)\n            record = dict(step=i, timestep=int(t), active=False, accepted_inner_steps=0)\n            latent = base.prev_sample.detach()\n            clean = base.pred_original_sample.detach()\n            if method == \'redenoise_only\' and hasattr(objective, \'record_baseline\'):\n                with torch.no_grad():objective.record_baseline(i, decode(clean))\n            if method == guided_name and i/max(len(timesteps)-1, 1) >= .2:\n                if hasattr(objective, \'begin_step\'):objective.begin_step(i)\n                if hasattr(objective, \'maybe_refresh\'):\n                    with torch.no_grad():\n                        refresh = objective.maybe_refresh(decode(clean), i)\n                    if refresh is not None:\n                        record[\'constraint_refresh\'] = refresh\n                        for name in (\'erase\', \'restore\'):\n                            mask = getattr(objective, name)[0, 0].detach().cpu().numpy()\n                            Image.fromarray(np.rint(mask*255).clip(0, 255).astype(\'uint8\')).save(out/f\'{name}_step_{i:03d}.png\')\n                        if hasattr(objective, \'export_state\'):\n                            objective.export_state(out/f\'ownership_step_{i:03d}.npz\')\n                clean, result = optimize_clean(clean, decode, objective,\n                    iterations=inner_steps if i < len(timesteps)-1 else inner_steps*2,\n                    step_ratio=step_ratio)\n                # DDIM eta=0: x_(t-1) = sqrt(a_prev)*x0 + sqrt(1-a_prev)*epsilon.\n                # Hold the predicted epsilon fixed and change the clean component.\n                prev_t = int(t)-scheduler.config.num_train_timesteps//scheduler.num_inference_steps\n                alpha = (scheduler.alphas_cumprod[prev_t] if prev_t >= 0\n                         else scheduler.final_alpha_cumprod).to(latent)\n                latent = latent+alpha.sqrt()*(clean-base.pred_original_sample)\n                record.update(active=True, **result)\n            latent = latent.detach()\n            trace.append(record)\n            if i % 4 == 0 or i == len(timesteps)-1:\n                save_image(clean, out/f\'{method}_step_{i:03d}.png\')\n            if progress:\n                progress(method, i+1, len(timesteps), record)\n        save_image(latent, out/f\'{method}.png\')\n        with torch.no_grad():\n            evaluate = getattr(objective, \'evaluate_fixed\', objective)\n            loss, values = evaluate(decode(latent))\n        metrics[method] = dict(loss=float(loss), **values)\n        if hasattr(objective, \'finish_arm\'):\n            with torch.no_grad():objective.finish_arm(method, decode(latent))\n        traces[method] = trace\n    with torch.no_grad():\n        loss, values = getattr(objective, \'evaluate_fixed\', objective)(objective.source)\n    metrics[\'input\'] = dict(loss=float(loss), **values)\n    report = dict(metadata=metadata, model_id=model_id, revision=revision,\n        steps=steps, actual_steps=len(timesteps), strength=strength, seed=seed,\n        cfg_scale=cfg_scale, inner_steps=inner_steps, step_ratio=step_ratio,\n        initial_latent_sha256=hashlib.sha256(initial.cpu().numpy().tobytes()).hexdigest(),\n        paired_initial_latent=True, guidance_target=guidance_mode,\n        constraints=getattr(objective, \'metadata\', None),\n        constraint_history=getattr(objective, \'history\', None),\n        target_pixels_pasted_into_output=False, metrics=metrics, traces=traces,\n        metric_note=(\'Fixed ink constraints and source preservation; not GT accuracy or HSQM score\'\n                     if guidance_mode in (\'ink_constraints\', \'adaptive_ink\') else \'Candidate agreement, not ground-truth accuracy or HSQM score\'))\n    if hasattr(objective, \'select_result\'):\n        import shutil\n        selection = objective.select_result(guided_name)\n        report[\'selection\'] = selection\n        shutil.copy2(out/selection[\'file\'], out/\'selected.png\')\n    (out/\'report.json\').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    del pipe\n    torch.cuda.empty_cache()\n    return report\n', 'stroke_model/adaptive_ink_constraints.py': '"""Frozen stroke identities, per-stroke bands, periodically refreshed ownership.\n\nGeometry is anchored to the initial fitted strokes, not re-fitted to new errors.\nExtractor softmax is detached; only image ink losses backpropagate through VAE.\n"""\nimport hashlib\nimport io\nimport json\nimport zipfile\n\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\nfrom scipy.ndimage import distance_transform_edt, binary_dilation\nfrom skimage.morphology import skeletonize\n\nfrom diffusion_model.fitted_candidate_diffusion import InkConstraintObjective, CandidateObjective\nfrom .stroke_reference_fit import _component, principal_path\n\n\ndef checkpoint_hash(path):\n    digest = hashlib.sha256()\n    with open(path, \'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024*1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef load_stroke_bundle(archive, case_id):\n    """Only known ZIP members are read, no recorded absolute paths are followed."""\n    with zipfile.ZipFile(archive) as z:\n        prefix = \'\'\n        if \'experiment_info.json\' in z.namelist():\n            info = json.loads(z.read(\'experiment_info.json\'))\n        else:\n            metadata = json.loads(z.read(\'report.json\'))[\'metadata\']\n            if metadata[\'case\'][\'case_id\'] != case_id:raise ValueError(\'Saved bundle CASE_ID mismatch\')\n            info = dict(checkpoint_sha256=metadata[\'checkpoint_sha256\'])\n            prefix = \'stroke_bundle/\'\n        manifest = json.loads(z.read(prefix+\'source_conditions/conditions.json\'))\n        records = []\n        if len(manifest) != 4:\n            raise ValueError(\'Exactly four reference slots required\')\n        for i, entry in enumerate(manifest):\n            name = entry[\'path\'].replace(\'\\\\\', \'/\').rsplit(\'/\', 1)[-1]\n            with np.load(io.BytesIO(z.read(prefix+\'source_conditions/\'+name)), allow_pickle=False) as d:\n                reference, strokes = d[\'reference\'].copy(), d[\'strokes\'].copy()\n            with np.load(io.BytesIO(z.read(f\'{prefix}{case_id}/warped_slot_{i+1}.npz\')), allow_pickle=False) as d:\n                fitted = d[\'warped_strokes\'].copy()\n            if strokes.shape != fitted.shape or reference.shape != (1, *strokes.shape[-2:]):\n                raise ValueError(\'Reference/fitted stroke shape mismatch\')\n            records.append(dict(reference=reference, strokes=strokes, fitted=fitted))\n        return records, info[\'checkpoint_sha256\']\n\n\nclass AdaptiveInkConstraints(InkConstraintObjective):\n    def __init__(self, source, extractor, records, *, refresh_every=3, enable_restore=False,\n                 margin_ratio=.2, margin_min=.6, margin_max=1.8,\n                 ownership_blend=.35, mask_blend=.5, preserve_weight=.5, continuity_weight=.2):\n        if source.shape[:2] != (1, 3) or len(records) != 4:\n            raise ValueError(\'One RGB sheet and four stroke records required\')\n        if type(refresh_every) is not int or refresh_every < 1 or not 0 < ownership_blend <= 1 or not 0 < mask_blend <= 1:\n            raise ValueError(\'Invalid refresh schedule/blending\')\n        if not 0 <= margin_ratio or not 0 < margin_min <= margin_max or preserve_weight < 0 or continuity_weight < 0:\n            raise ValueError(\'Invalid stroke bands/weights\')\n        self.source = source.detach()\n        self.extractor = extractor.eval().requires_grad_(False)\n        self.refresh_every, self.ownership_blend, self.mask_blend = refresh_every, ownership_blend, mask_blend\n        self.enable_restore = enable_restore\n        self.preserve_weight, self.continuity_weight = preserve_weight, continuity_weight\n        self.background_ink, self.required_ink = .015, .7\n        self.records, self.bands, self.history = [], [], []\n        h, w = source.shape[-2:]\n        if h % 2 or w % 2:\n            raise ValueError(\'Equal 2x2 slots require even sheet dimensions\')\n        boxes = [(0, 0, w//2, h//2), (w//2, 0, w, h//2),\n                 (0, h//2, w//2, h), (w//2, h//2, w, h)]\n        source_ink = self.ink(source)\n        # Identity k always refers to input canonical stroke k, never sorted/merged.\n        for slot, (raw, box) in enumerate(zip(records, boxes)):\n            ref = torch.as_tensor(raw[\'reference\'], device=source.device, dtype=torch.float32)[None]\n            strokes = torch.as_tensor(raw[\'strokes\'], device=source.device, dtype=torch.float32)[None]\n            fitted = np.asarray(raw[\'fitted\'], np.float32)\n            crop = self.crop(source_ink, box, fitted.shape[-2:])\n            probabilities = self.predict(crop, ref, strokes)\n            observed = crop[0, 0].cpu().numpy()\n            distances, radii, margins, cores, source_bodies = [], [], [], [], []\n            for k, mask in enumerate(fitted):\n                binary = mask > .35\n                skeleton = skeletonize(binary)\n                if not skeleton.any():\n                    raise ValueError(f\'Slot {slot+1}, stroke {k+1}: empty fitted path\')\n                inside = distance_transform_edt(binary)\n                distance, nearest = distance_transform_edt(~skeleton, return_indices=True)\n                expected = max(float(np.median(inside[skeleton])), .75)\n                body = (observed > .35) & (probabilities[k] > .35)\n                width_samples = distance_transform_edt(body)[skeleton & body]\n                # Only samples on the anchored main path calibrate width. Extras\n                # off the path cannot enlarge the allowed corridor arbitrarily.\n                observed_half = float(np.median(width_samples)) if width_samples.size >= 4 else expected\n                half = float(np.clip(observed_half, .75*expected, 1.25*expected))\n                local_radius = np.maximum(inside[tuple(nearest)], .75)*(half/expected)\n                margin = float(np.clip(margin_ratio*2*half, margin_min, margin_max))\n                trusted_core = skeleton & (observed > .35)\n                source_body = np.zeros_like(binary)\n                path_protected = False\n                try:\n                    component, _ = _component(probabilities[k]*observed, hint=mask)\n                    path, path_info = principal_path(component, n=128)\n                    length = float(np.linalg.norm(np.diff(path, axis=0), axis=1).sum())\n                    ratio = length/max(int(skeleton.sum()), 1)\n                    if not path_info[\'closed\'] and .65 <= ratio <= 1.5 and path_info[\'path_coverage\'] > .65:\n                        dense = np.stack([np.interp(np.linspace(0, 1, 512), np.linspace(0, 1, len(path)), path[:, j]) for j in range(2)], 1)\n                        xy = np.rint(dense).astype(int)\n                        xy[:, 0] = xy[:, 0].clip(0, binary.shape[1]-1)\n                        xy[:, 1] = xy[:, 1].clip(0, binary.shape[0]-1)\n                        source_path = np.zeros_like(binary);source_path[xy[:, 1], xy[:, 0]] = True\n                        if float(np.median(distance[source_path])) <= expected+margin:\n                            # Protect the original main body, not the smooth\n                            # candidate outline. Cap radius to exclude side branches.\n                            source_body = (distance_transform_edt(~source_path) <= 1.1*half) & (observed > .35)\n                            trusted_core = source_path & (observed > .35)\n                            path_protected = True\n                except ValueError:\n                    pass  # Explicitly reported fallback to anchored-path protection.\n                distances.append(distance); radii.append(local_radius); margins.append(margin); cores.append(trusted_core)\n                source_bodies.append(source_body)\n                self.bands.append(dict(slot=slot+1, stroke=k+1, half_width_model=half,\n                    tolerance_model=margin, tolerance_sheet_x=margin*(box[2]-box[0])/fitted.shape[-1],\n                    tolerance_sheet_y=margin*(box[3]-box[1])/fitted.shape[-2],\n                    trusted_core_pixels=int(trusted_core.sum()), original_main_path_protected=path_protected))\n            self.records.append(dict(reference=ref, strokes=strokes, box=box,\n                size=fitted.shape[-2:], probabilities=probabilities,\n                distances=np.stack(distances), radii=np.stack(radii), margins=np.asarray(margins)[:, None, None],\n                cores=np.stack(cores), original_bodies=np.stack(source_bodies),\n                skeletons=np.stack([skeletonize(s > .35) for s in fitted])))\n        self.erase = torch.zeros_like(source_ink)\n        self.restore = torch.zeros_like(source_ink)\n        self.protected_cores = []\n        for record in self.records:\n            for core in record[\'cores\']:\n                canvas = torch.zeros_like(source_ink)\n                self.place(canvas, self.tensor(core.astype(np.float32)), record[\'box\'])\n                self.protected_cores.append(canvas)\n        self.refresh(source, step=-1, initial=True)\n        self.fixed = dict(erase=self.erase.clone(), restore=self.restore.clone(), preserve=self.preserve.clone())\n        self.metadata = dict(kind=\'adaptive_stroke_ink_constraints\', candidate_rgb_matching=False,\n            enable_restore=enable_restore, refresh_every=refresh_every, stroke_identities_frozen=True,\n            geometry_frozen=True, width_calibration=\'robust trusted main-path samples, frozen\',\n            ownership_backward=False, bands=self.bands, preserve_weight=preserve_weight,\n            continuity_weight=continuity_weight, masks_frozen_for_entire_pass=False)\n\n    def tensor(self, a):\n        return torch.as_tensor(np.asarray(a).copy(), dtype=torch.float32, device=self.source.device)[None, None]\n\n    @staticmethod\n    def crop(image, box, size):\n        x0, y0, x1, y1 = box\n        return F.interpolate(image[..., y0:y1, x0:x1], size=size, mode=\'bilinear\', align_corners=False)\n\n    @staticmethod\n    def place(canvas, patch, box):\n        x0, y0, x1, y1 = box\n        canvas[..., y0:y1, x0:x1] = F.interpolate(patch, size=(y1-y0, x1-x0), mode=\'bilinear\', align_corners=False)\n\n    @torch.no_grad()\n    def predict(self, ink, reference, strokes):\n        p = self.extractor(ink, reference, strokes)[\'probabilities\'][0].float()\n        if p.shape != strokes.shape[1:] or not torch.isfinite(p).all() or (p < 0).any():\n            raise ValueError(\'Invalid extractor ownership probabilities\')\n        p = p/p.sum(0, keepdim=True).clamp_min(1e-8)\n        return p.cpu().numpy()\n\n    @torch.no_grad()\n    def refresh(self, image, *, step, initial=False):\n        ink = self.ink(image).clamp(0, 1)\n        erase, restore = torch.zeros_like(ink), torch.zeros_like(ink)\n        rows = []\n        for slot, r in enumerate(self.records):\n            crop = self.crop(ink, r[\'box\'], r[\'size\'])\n            current = self.predict(crop, r[\'reference\'], r[\'strokes\'])\n            p = current if initial else (1-self.ownership_blend)*r[\'probabilities\']+self.ownership_blend*current\n            r[\'probabilities\'] = p\n            observed = crop[0, 0].cpu().numpy()\n            excess = r[\'distances\']-r[\'radii\']\n            outside = np.clip((excess-r[\'margins\'])/np.maximum(r[\'margins\'], .1), 0, 1)\n            # Protect every other legitimate stroke at intersections too.\n            safe_body = (excess <= r[\'margins\']).any(0) | r[\'original_bodies\'].any(0)\n            responsibility = (p*outside).sum(0)\n            # Existing faint residuals keep their old mask. New regions require\n            # more than faint decoder noise before they can expand deletion.\n            seeds = (observed > (.25 if initial else .12)) & (responsibility > .4) & ~safe_body\n            grown = binary_dilation(seeds, iterations=1)\n            weight = grown*responsibility*(~safe_body)\n            missing = r[\'skeletons\'].any(0) & (distance_transform_edt(~(observed > .2)) > 1.)\n            restore_weight = missing.astype(np.float32) if self.enable_restore else np.zeros_like(weight)\n            weight[restore_weight > 0] = 0\n            self.place(erase, self.tensor(weight), r[\'box\'])\n            self.place(restore, self.tensor(restore_weight), r[\'box\'])\n            for k in range(len(p)):\n                rows.append(dict(slot=slot+1, stroke=k+1,\n                    assigned_ink=float((p[k]*observed).sum()),\n                    outside_ink=float((p[k]*observed*outside[k]*(~safe_body)).sum())))\n        # Retain previous deletion support: fading errors must not fall out of\n        # the mask or become protected just because they were partly erased.\n        self.erase = erase if initial else torch.maximum(self.erase, self.mask_blend*erase)\n        self.restore = restore if initial else (1-self.mask_blend)*self.restore+self.mask_blend*restore\n        self.erase = self.erase*(self.restore <= 0).float()\n        self.edit = torch.maximum(self.erase, self.restore)\n        self.preserve = 1-F.max_pool2d((self.edit > .01).float(), 5, 1, 2)\n        row = dict(step=step, erase_pixels=int((self.erase > .01).sum()),\n                   restore_pixels=int((self.restore > .01).sum()), per_stroke=rows)\n        self.history.append(row)\n        return row\n\n    def maybe_refresh(self, image, step):\n        return self.refresh(image, step=step) if step % self.refresh_every == 0 else None\n\n    def export_state(self, path):\n        arrays = {f\'slot_{i+1}_probabilities\': r[\'probabilities\'] for i, r in enumerate(self.records)}\n        arrays.update(erase=self.erase.detach().cpu().numpy(), restore=self.restore.detach().cpu().numpy())\n        np.savez_compressed(path, **arrays)\n\n    def __call__(self, image):\n        loss, details = super().__call__(image)\n        ink = self.ink(image)\n        # Protect initially present main paths, averaged per stroke, not per sheet.\n        continuity = [CandidateObjective.mean_region(F.relu(.65-ink), core)\n                      for core in self.protected_cores if float(core.sum()) > 0]\n        value = torch.stack(continuity).mean() if continuity else loss*0\n        details[\'trusted_path_deficit\'] = float(value.detach())\n        return loss+self.continuity_weight*value, details\n\n    def evaluate_fixed(self, image):\n        current = dict(erase=self.erase, restore=self.restore, preserve=self.preserve)\n        try:\n            for key, value in self.fixed.items():\n                setattr(self, key, value)\n            return self(image)\n        finally:\n            for key, value in current.items():\n                setattr(self, key, value)\n', 'stroke_model/guarded_ink_constraints.py': '"""Conservative release policy: static base, consensus updates, stroke guards.\n\nNo quality guarantees: rejected edits/fallbacks are explicitly reported. All arms\nremain available; selected.png is copied from one complete output, never composited.\n"""\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\nfrom scipy.ndimage import distance_transform_edt\n\nfrom diffusion_model.fitted_candidate_diffusion import InkConstraintObjective\nfrom .adaptive_ink_constraints import AdaptiveInkConstraints\nfrom .stroke_reference_fit import principal_path\n\n\nclass GuardedInkConstraints(AdaptiveInkConstraints):\n    def __init__(self, source, candidate, extractor, records, *, enable_restore=False,\n                 confirmations=2, refresh_every=3, mse_slack=.001,\n                 local_slack=.025, ink_drop_slack=.02):\n        if type(confirmations) is not int or confirmations < 2:\n            raise ValueError(\'At least two confirmations required\')\n        if any(not np.isfinite(v) or v < 0 for v in (mse_slack, local_slack, ink_drop_slack)):\n            raise ValueError(\'Invalid preservation limits\')\n        super().__init__(source, extractor, records, enable_restore=enable_restore, refresh_every=refresh_every)\n        self.confirmations = confirmations\n        self.slacks = source.new_tensor([mse_slack, local_slack, ink_drop_slack])\n        # The successful sparse fixed-corridor method is the default foundation.\n        base = InkConstraintObjective(source, candidate, enable_restore=enable_restore)\n        self.base_erase, self.base_restore = base.erase.clone(), base.restore.clone()\n        self.branch_masks = []\n        self.uncertain = torch.zeros_like(self.base_erase)\n        source_ink = self.ink(source)\n        for r in self.records:\n            observed = self.crop(source_ink, r[\'box\'], r[\'size\'])[0, 0].cpu().numpy()\n            masks = []\n            for k in range(len(r[\'probabilities\'])):\n                proposal = self.straight_branch(r, k, observed)\n                masks.append(proposal)\n                canvas = torch.zeros_like(self.uncertain)\n                self.place(canvas, self.tensor(proposal), r[\'box\'])\n                self.uncertain = torch.maximum(self.uncertain, canvas)\n            self.branch_masks.append(np.stack(masks))\n        self.pending = torch.zeros_like(self.base_erase, dtype=torch.int32)\n        self.dynamic = torch.zeros_like(self.base_erase)\n        excluded = F.max_pool2d(((self.base_erase > .01) | (self.base_restore > .01) |\n                                 (self.uncertain > .05)).float(), 7, 1, 3)\n        self.healthy_groups = []\n        for r in self.records:\n            for p in r[\'probabilities\']:\n                canvas = torch.zeros_like(source_ink)\n                self.place(canvas, self.tensor(p), r[\'box\'])\n                group = canvas*(source_ink > .2).float()*(1-excluded)\n                if float(group.sum()) > 1:self.healthy_groups.append(group)\n        self.groups = torch.cat(self.healthy_groups, 0) if self.healthy_groups else torch.zeros_like(source_ink)\n        self.healthy_union = self.groups.sum(0, keepdim=True).clamp(0, 1)\n        self.baseline, self.arm_states = {}, {}\n        self.current_step = None\n        self.erase, self.restore = self.base_erase.clone(), self.base_restore.clone()\n        self.update_support()\n        self.fixed = dict(erase=self.erase.clone(), restore=self.restore.clone(), preserve=self.preserve.clone())\n        self.fixed_error = torch.maximum(self.erase, self.restore).clone()\n        self.selection_erase = torch.maximum(self.base_erase, self.uncertain).clone()\n        self.history = [dict(step=-1, erase_pixels=int((self.erase > .01).sum()),\n                             restore_pixels=int((self.restore > .01).sum()), policy=\'static_base\')]\n        self.metadata.update(kind=\'guarded_ink_release\', confirmations=confirmations,\n            fixed_base=True, dynamic_masks_revocable=True, normal_groups_frozen=True,\n            mse_slack=mse_slack, local_slack=local_slack, ink_drop_slack=ink_drop_slack,\n            original_prototype_rgb_target=False, straight_branch_proposal_pixels=int((self.uncertain > .05).sum()))\n        self.ready = True\n\n    def __call__(self, image):\n        loss, details = super().__call__(image)\n        if not getattr(self, \'ready\', False):return loss, details\n        # Counter VAE leakage before a hard guard rejects the proposal. Region-\n        # normalized stroke errors are not diluted by a large white background.\n        error = (image-self.source).square().mean(1, keepdim=True)\n        mass = self.groups.sum((1, 2, 3)).clamp_min(1)\n        stroke_error = (error*self.groups).sum((1, 2, 3))/mass\n        pooled_mass = F.avg_pool2d(self.groups, 5, 1, 2)\n        local = F.avg_pool2d(error*self.groups, 5, 1, 2)/pooled_mass.clamp_min(.1)\n        boundary = local.flatten(1).amax(1).mean()\n        details[\'normal_stroke_mse\'] = float(stroke_error.mean().detach())\n        details[\'normal_local_mse\'] = float(boundary.detach())\n        return loss+stroke_error.mean()+boundary, details\n\n    def straight_branch(self, r, k, observed):\n        """Propose side bulges of a straight prototype; do not target its caps."""\n        empty = np.zeros_like(observed, dtype=np.float32)\n        try:\n            canonical, _ = principal_path(r[\'strokes\'][0, k].cpu().numpy() > .2)\n            chord = np.linalg.norm(canonical[-1]-canonical[0])\n            length = np.linalg.norm(np.diff(canonical, axis=0), axis=1).sum()\n            if chord/max(length, 1e-6) < .98:return empty\n            path, _ = principal_path(r[\'skeletons\'][k])\n        except ValueError:\n            return empty\n        a, b = path[0], path[-1]\n        axis = b-a;length = float(np.linalg.norm(axis))\n        if length < 5:return empty\n        axis /= length;normal = np.array([-axis[1], axis[0]])\n        yy, xx = np.indices(observed.shape)\n        xy = np.stack((xx, yy), -1)-a\n        along, across = xy@axis, xy@normal\n        body = (observed > .35) & (r[\'probabilities\'][k] > .5)\n        widths = distance_transform_edt(body)[r[\'skeletons\'][k] & body & (along > .2*length) & (along < .8*length)]\n        if widths.size < 4:return empty\n        calibrated = float(np.median(r[\'radii\'][k][r[\'skeletons\'][k]]))\n        radius = float(np.clip(np.quantile(widths, .25), .65*calibrated, calibrated))\n        # Restrict to the middle; irregular stroke caps are deliberately untouched.\n        excess = np.clip((np.abs(across)-radius-.6)/max(radius*.4, .5), 0, 1)\n        return (excess*(along > .1*length)*(along < .9*length)*body).astype(np.float32)\n\n    def update_support(self):\n        self.edit = torch.maximum(self.erase, self.restore)\n        self.preserve = 1-F.max_pool2d((self.edit > .01).float(), 5, 1, 2)\n\n    @torch.no_grad()\n    def refresh(self, image, *, step, initial=False):\n        if not getattr(self, \'ready\', False):return super().refresh(image, step=step, initial=initial)\n        old_erase, old_restore = self.erase, self.restore\n        self.erase = torch.zeros_like(old_erase);self.restore = torch.zeros_like(old_restore)\n        row = super().refresh(image, step=step)\n        raw = self.erase.clone()\n        self.erase, self.restore = old_erase, old_restore\n        # Branch proposals inside a broad geometric corridor need separate evidence.\n        current_ink = self.ink(image).clamp(0, 1)\n        branches = torch.zeros_like(raw)\n        for r, masks in zip(self.records, self.branch_masks):\n            crop = self.crop(current_ink, r[\'box\'], r[\'size\'])[0, 0].cpu().numpy()\n            weight = (masks*r[\'probabilities\']).sum(0)*(crop > .12)\n            self.place(branches, self.tensor(weight), r[\'box\'])\n        raw = torch.maximum(raw, branches)\n        # Existing healthy ink is not reclassified merely to make a new mask fit.\n        raw = raw*(self.healthy_union < .1).float()\n        positive = raw > .05\n        self.pending = torch.where(positive, self.pending+1, torch.zeros_like(self.pending))\n        confirmed = self.pending >= self.confirmations\n        # New masks need repeated evidence. Lost evidence can revoke old additions.\n        self.dynamic = torch.where(confirmed, raw, self.dynamic*.6)\n        self.dynamic[self.dynamic < .02] = 0\n        self.erase = torch.maximum(self.base_erase, self.dynamic)\n        self.restore = self.base_restore.clone()\n        self.erase = self.erase*(self.restore <= 0).float()\n        self.update_support()\n        row.update(erase_pixels=int((self.erase > .01).sum()),\n                   confirmed_pixels=int(confirmed.sum()), pending_pixels=int((self.pending == 1).sum()))\n        return row\n\n    @torch.no_grad()\n    def guard_state(self, image):\n        rgb = image.clamp(0, 1)\n        error = (rgb-self.source).square().mean(1, keepdim=True)\n        weights = self.groups\n        mass = weights.sum((1, 2, 3)).clamp_min(1)\n        mse = (error*weights).sum((1, 2, 3))/mass\n        pooled_mass = F.avg_pool2d(weights, 5, 1, 2)\n        local = F.avg_pool2d(error*weights, 5, 1, 2)/pooled_mass.clamp_min(.1)\n        peak = local.flatten(1).amax(1)\n        drop = (F.relu(self.ink(self.source)-self.ink(rgb))*weights).sum((1, 2, 3))/mass\n        return torch.stack((mse, peak, drop), -1)\n\n    def record_baseline(self, step, image):self.baseline[step] = self.guard_state(image)\n    def begin_step(self, step):self.current_step = step\n\n    def check_proposal(self, before, after):\n        if self.current_step not in self.baseline:raise RuntimeError(\'Matched baseline guard missing\')\n        # An already drifting anchor may improve but must not get any worse.\n        bound = torch.maximum(self.baseline[self.current_step]+self.slacks, self.guard_state(before))\n        return bool((self.guard_state(after) <= bound+1e-7).all())\n\n    def finish_arm(self, name, image):\n        error = self.error_value(image)\n        self.arm_states[name] = dict(guard=self.guard_state(image), error=error)\n\n    def error_value(self, image):\n        ink = self.ink(image)\n        erase = ((ink-self.background_ink).clamp_min(0)*self.selection_erase).sum()/self.selection_erase.sum().clamp_min(1)\n        restore = ((self.required_ink-ink).clamp_min(0)*self.base_restore).sum()/self.base_restore.sum().clamp_min(1)\n        return float(erase+restore)\n\n    def select_result(self, guided_name):\n        base, guided = self.arm_states[\'redenoise_only\'], self.arm_states[guided_name]\n        normal_ok = bool((guided[\'guard\'] <= base[\'guard\']+self.slacks+1e-7).all())\n        initial = self.error_value(self.source)\n        improved = guided[\'error\'] < min(base[\'error\'], initial)-1e-5\n        accepted = normal_ok and improved\n        return dict(file=guided_name+\'.png\' if accepted else \'input.png\',\n            status=\'guided_passed_internal_checks\' if accepted else \'original_retained\',\n            normal_guard_passed=normal_ok, fixed_error_improved=improved,\n            original_error=initial, baseline_error=base[\'error\'], guided_error=guided[\'error\'],\n            guard_worst_guided=guided[\'guard\'].amax(0).cpu().tolist(),\n            guard_worst_baseline=base[\'guard\'].amax(0).cpu().tolist(),\n            note=\'Internal heuristic checks, not verified visual correctness; all raw arms retained\')\n', 'stroke_model/reference_model.py': '"""Reference-conditioned V2. Inputs: white ink on black, Bx1xHxW in [0,1].\n\nNo thresholding, CPU conversion or detached image features in the forward path.\nOld checkpoints are deliberately not compatible with this new architecture.\n"""\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\ndef block(cin, cout, stride=1):\n    return nn.Sequential(nn.Conv2d(cin, cout, 3, stride=stride, padding=1),\n                         nn.GroupNorm(4, cout), nn.GELU())\n\n\ndef resize(x, size):\n    return F.interpolate(x, size=size, mode="bilinear", align_corners=False)\n\n\ndef warp(strokes, flow):\n    b, _, h, w = strokes.shape\n    ys = (torch.arange(h, device=strokes.device, dtype=strokes.dtype) + .5) * 2 / h - 1\n    xs = (torch.arange(w, device=strokes.device, dtype=strokes.dtype) + .5) * 2 / w - 1\n    y, x = torch.meshgrid(ys, xs, indexing="ij")\n    grid = torch.stack((x, y), -1)[None].expand(b, -1, -1, -1)\n    offset = torch.stack((flow[:, 0] * 2 / w, flow[:, 1] * 2 / h), -1)\n    return F.grid_sample(strokes, grid + offset, align_corners=False)\n\n\nclass ReferenceStrokeModel(nn.Module):\n    def __init__(self, image_size=128, width=64, pretrained=True,\n                 backbone=None, channels=None, channels_last=False, stroke_chunk_size=4):\n        super().__init__()\n        if width % 4:\n            raise ValueError("width must be divisible by 4")\n        mean, std = (.485, .456, .406), (.229, .224, .225)\n        if backbone is None:\n            import timm\n            backbone = timm.create_model(\n                "swin_tiny_patch4_window7_224", pretrained=pretrained,\n                features_only=True, out_indices=(0, 1, 2, 3),\n                in_chans=3, img_size=image_size, drop_path_rate=0.)\n            channels = backbone.feature_info.channels()\n            channels_last = True  # timm Swin features are NHWC.\n            mean = backbone.pretrained_cfg.get("mean", mean)\n            std = backbone.pretrained_cfg.get("std", std)\n        if not channels:\n            raise ValueError("Injected backbone requires its feature channels")\n        self.backbone = backbone\n        self.stroke_chunk_size = max(1, int(stroke_chunk_size))\n        self.channels_last = channels_last\n        self.backbone_frozen = True\n        self.backbone.requires_grad_(False)\n        self.backbone.eval()\n        self.register_buffer("rgb_mean", torch.tensor(mean).view(1, 3, 1, 1))\n        self.register_buffer("rgb_std", torch.tensor(std).view(1, 3, 1, 1))\n        self.lateral = nn.ModuleList(nn.Conv2d(c, width, 1) for c in channels)\n        self.smooth = nn.ModuleList(block(width, width) for _ in channels)\n        self.detail = block(1, width)  # Full-resolution branch for thin strokes.\n        self.fuse = block(width * 2, width)\n        self.flow_net = nn.Sequential(block(2, 16, 2), block(16, 32, 2),\n                                      nn.Conv2d(32, 2, 3, padding=1))\n        nn.init.zeros_(self.flow_net[-1].weight)\n        nn.init.zeros_(self.flow_net[-1].bias)\n        # Shared across stroke IDs; full glyph + selected stroke + xy context.\n        self.reference = nn.Sequential(block(4, 16, 2), block(16, width, 2))\n        self.head = nn.Sequential(block(width * 2 + 4, width),\n                                  nn.Conv2d(width, 1, 1))\n\n    def train(self, mode=True):\n        super().train(mode)\n        if self.backbone_frozen:\n            self.backbone.eval()\n        return self\n\n    def forward(self, image, reference, strokes, active=None):\n        if image.ndim != 4 or image.shape[1] != 1 or reference.shape != image.shape:\n            raise ValueError("image/reference must have matching Bx1xHxW shapes")\n        b, _, h, w = image.shape\n        if strokes.ndim != 4 or strokes.shape[0] != b or strokes.shape[-2:] != (h, w):\n            raise ValueError("strokes must have shape BxKxHxW")\n        k = strokes.shape[1]\n        if active is None:\n            active = strokes.flatten(2).sum(-1) > 0  # Fixed reference metadata only.\n        if active.shape != (b, k) or active.dtype != torch.bool or not active.any(1).all():\n            raise ValueError("Each sample needs at least one active reference stroke")\n        rgb = (1 - image).expand(-1, 3, -1, -1)\n        feats = self.backbone((rgb - self.rgb_mean) / self.rgb_std)\n        if self.channels_last:\n            feats = [f.permute(0, 3, 1, 2) for f in feats]\n        pyramid = None\n        for i in reversed(range(len(feats))):\n            lateral = self.lateral[i](feats[i])\n            pyramid = lateral if pyramid is None else lateral + resize(pyramid, lateral.shape[-2:])\n            pyramid = self.smooth[i](pyramid)\n        features = self.fuse(torch.cat((resize(pyramid, (h, w)), self.detail(image)), 1))\n        flow = 12 * resize(self.flow_net(torch.cat((reference, image), 1)), (h, w)).tanh()\n        hints = warp(strokes, flow)\n        yy, xx = torch.meshgrid(torch.linspace(-1, 1, h, device=image.device, dtype=image.dtype),\n                                torch.linspace(-1, 1, w, device=image.device, dtype=image.dtype), indexing="ij")\n        xy = torch.stack((xx, yy))[None].expand(b, -1, -1, -1)\n        logits = self.score_strokes(features, reference, strokes, xy, flow, hints, image)\n        logits = logits.float().masked_fill(~active[:, :, None, None], -torch.inf)\n        probabilities = logits.softmax(1) * active[:, :, None, None]\n        probabilities = probabilities / probabilities.sum(1, keepdim=True)\n        return {"logits": logits, "probabilities": probabilities,\n                "masks": image * probabilities, "flow": flow, "hints": hints,\n                "active": active}\n\n    def score_strokes(self, features, reference, strokes, xy, flow, hints, image):\n        """Same head algebra, batching strokes and computing shared convolution once.\n\n        GroupNorm is per sample, so stroke batching does not mix stroke statistics.\n        Existing V2 state_dicts remain compatible; input gradients remain connected.\n        """\n        b, k, h, w = strokes.shape\n        width = features.shape[1]\n        conv = self.head[0][0]\n        shared_weight = torch.cat((conv.weight[:, :width], conv.weight[:, 2*width+1:]), 1)\n        shared = F.conv2d(torch.cat((features, image, xy), 1), shared_weight, conv.bias, padding=1)\n        outputs = []\n        for start in range(0, k, self.stroke_chunk_size):\n            n = min(self.stroke_chunk_size, k-start)\n            def repeat(t):\n                return t[:, None].expand(-1, n, -1, -1, -1).reshape(b*n, *t.shape[1:])\n            selected = strokes[:, start:start+n].reshape(b*n, 1, h, w)\n            r = self.reference(torch.cat((repeat(reference), selected, repeat(xy)), 1))\n            r = warp(resize(r, (h, w)), repeat(flow))\n            condition = torch.cat((r, hints[:, start:start+n].reshape(b*n, 1, h, w)), 1)\n            x = F.conv2d(condition, conv.weight[:, width:2*width+1], padding=1) + repeat(shared)\n            x = self.head[0][2](self.head[0][1](x))\n            outputs.append(self.head[1](x).reshape(b, n, h, w))\n        return torch.cat(outputs, 1)\n\n\ndef exclusive_masks(output, foreground):\n    """Export only: non-differentiable exact partition of the supplied foreground."""\n    expected = output[\'probabilities\'][:, :1].shape\n    if foreground.shape != expected:\n        raise ValueError(\'foreground must have shape Bx1xHxW\')\n    labels = output["probabilities"].argmax(1)\n    masks = F.one_hot(labels, output["probabilities"].shape[1]).permute(0, 3, 1, 2)\n    return masks.bool() & foreground.bool()\n\n\ndef stroke_moments(masks, eps=1e-6):\n    """Area fraction, centroid x/y and xx/xy/yy covariance in normalized coordinates.\n\n    Near-empty strokes have unreliable geometry: callers must mask/inspect them.\n    """\n    h, w = masks.shape[-2:]\n    y, x = torch.meshgrid(torch.linspace(-1, 1, h, device=masks.device, dtype=masks.dtype),\n                          torch.linspace(-1, 1, w, device=masks.device, dtype=masks.dtype), indexing="ij")\n    mass = masks.sum((-2, -1))\n    weights = masks / mass.clamp_min(eps)[..., None, None]\n    cx, cy = (weights * x).sum((-2, -1)), (weights * y).sum((-2, -1))\n    dx, dy = x - cx[..., None, None], y - cy[..., None, None]\n    return torch.stack((mass / (h * w), cx, cy,\n                        (weights * dx.square()).sum((-2, -1)),\n                        (weights * dx * dy).sum((-2, -1)),\n                        (weights * dy.square()).sum((-2, -1))), -1)\n\n\ndef partition_loss(output, labels):\n    """labels: BxHxW, 0..K-1 for owned pixels; -1 background/unknown.\n\n    Unknown crossing pixels are excluded, not assigned pseudo-certainty.\n    Soft Dice averages only annotated, nonempty target strokes.\n    """\n    logits = output["logits"]\n    valid = labels >= 0\n    if labels.shape != logits.shape[:1] + logits.shape[2:]:\n        raise ValueError("labels must have shape BxHxW")\n    if not valid.any() or (labels < -1).any() or (labels >= logits.shape[1]).any():\n        raise ValueError("Need annotated pixels with labels in -1..K-1")\n    target = F.one_hot(labels.clamp_min(0).long(), logits.shape[1]).permute(0, 3, 1, 2)\n    target = target * valid[:, None]\n    if (target.bool() & ~output["active"][:, :, None, None]).any():\n        raise ValueError("Target refers to an inactive reference stroke")\n    ce = F.cross_entropy(logits, labels.long(), ignore_index=-1)\n    pred = output["probabilities"] * valid[:, None]\n    mass = target.sum((-2, -1))\n    dice = (2 * (pred * target).sum((-2, -1)) + 1e-6) / (pred.sum((-2, -1)) + mass + 1e-6)\n    dice_loss = (1 - dice)[mass > 0].mean()\n    flow = output["flow"]\n    smooth = (flow[..., 1:, :] - flow[..., :-1, :]).square().mean()\n    smooth = smooth + (flow[..., :, 1:] - flow[..., :, :-1]).square().mean()\n    return {"loss": ce + dice_loss + .001 * smooth, "ce": ce,\n            "dice_loss": dice_loss, "flow_smoothness": smooth}\n', 'stroke_model/stroke_reference_fit.py': '"""Fit valid reference strokes to extracted stroke positions, not their defects.\n\nInference-only geometry: principal skeleton paths discard short side branches;\nstraight references remain straight, curves receive a regularized deformation.\nThe original reference centerline is deformed and re-rendered at the observed\nwidth. Output is a candidate for inspection, not a ground-truth reconstruction.\n"""\nfrom dataclasses import dataclass, asdict\nimport heapq\nimport numpy as np\nfrom scipy.ndimage import label, distance_transform_edt, map_coordinates, affine_transform,gaussian_filter1d\nfrom scipy.spatial import cKDTree\nfrom skimage.morphology import skeletonize\n\n\n@dataclass(frozen=True)\nclass ReferenceFitConfig:\n    points: int=64\n    smoothness: float=12.\n    shape_prior: float=.15\n    max_displacement: float=16.\n    max_width: float=10.\n    min_width: float=1.\n    min_length_ratio: float=.65\n    max_length_ratio: float=1.6\n    junction_radius: float=2.5\n    junction_weight: float=8.\n    max_junction_shift: float=2.\n\n    def __post_init__(self):\n        if type(self.points) is not int or self.points<8:raise ValueError(\'At least eight path samples required\')\n        if any(not np.isfinite(v) or v<0 for v in asdict(self).values()):raise ValueError(\'Invalid fit configuration\')\n        if not self.min_width or self.max_width<self.min_width or not 0<self.min_length_ratio<self.max_length_ratio:\n            raise ValueError(\'Invalid width/length limits\')\n\n\ndef _unit(array):\n    array=np.asarray(array,np.float32)\n    if array.ndim!=2 or not np.isfinite(array).all() or array.min()<0 or array.max()>1:\n        raise ValueError(\'Expected a finite 2D ink mask in [0,1]\')\n    return array\n\n\ndef _component(mask,hint=None):\n    threshold=max(.1,min(.4,float(mask.max())*.4))\n    binary=mask>=threshold\n    labels,count=label(binary,np.ones((3,3)))\n    if not count:raise ValueError(\'Empty/low-confidence extracted stroke\')\n    distance=None\n    if hint is not None and hint.max()>.1:\n        distance=distance_transform_edt(hint<max(.1,float(hint.max())*.3))\n    scores=[]\n    for k in range(1,count+1):\n        region=labels==k;mass=float(mask[region].sum())\n        proximity=1. if distance is None else np.exp(-float(np.median(distance[region]))/20.)\n        scores.append(mass*proximity)\n    chosen=labels==(int(np.argmax(scores))+1)\n    return chosen,dict(components=count,selected_mass_fraction=float(mask[chosen].sum()/max(float(mask.sum()),1e-6)))\n\n\ndef _resample(path,n):\n    arc=np.r_[0,np.cumsum(np.linalg.norm(np.diff(path,axis=0),axis=1))]\n    if arc[-1]<1e-6:raise ValueError(\'Degenerate stroke path\')\n    keep=np.r_[True,np.diff(arc)>1e-8];arc=arc[keep];path=path[keep]\n    t=np.linspace(0,arc[-1],n)\n    return np.stack([np.interp(t,arc,path[:,i]) for i in range(2)],1)\n\n\ndef principal_path(binary,n=64):\n    """Longest geodesic path of the principal component; never reject spurs."""\n    skeleton=skeletonize(binary)\n    coordinates=np.argwhere(skeleton)\n    if len(coordinates)<2:raise ValueError(\'Insufficient centerline support\')\n    indices={tuple(yx):i for i,yx in enumerate(coordinates)}\n    graph=[[] for _ in coordinates]\n    for i,(y,x) in enumerate(coordinates):\n        for dy in (-1,0,1):\n            for dx in (-1,0,1):\n                if not (dx or dy):continue\n                j=indices.get((y+dy,x+dx))\n                if j is not None:graph[i].append((j,float(np.hypot(dx,dy))))\n    endpoints=[i for i,edges in enumerate(graph) if len(edges)==1]\n    def farthest(start):\n        distances=np.full(len(graph),np.inf);distances[start]=0.;parent=np.full(len(graph),-1,int)\n        queue=[(0.,start)]\n        while queue:\n            cost,i=heapq.heappop(queue)\n            if cost>distances[i]+1e-8:continue\n            for j,weight in graph[i]:\n                proposed=cost+weight\n                if proposed<distances[j]-1e-8:\n                    distances[j]=proposed;parent[j]=i;heapq.heappush(queue,(proposed,j))\n        candidates=endpoints if endpoints else list(range(len(graph)))\n        end=max(candidates,key=lambda j:distances[j] if np.isfinite(distances[j]) else -1)\n        return end,parent\n    first,_=farthest(endpoints[0] if endpoints else 0)\n    last,parent=farthest(first);order=[last]\n    while order[-1]!=first:\n        previous=parent[order[-1]]\n        if previous<0:raise ValueError(\'Disconnected centerline\')\n        order.append(int(previous))\n    path=coordinates[order,::-1].astype(float)\n    return _resample(path,n),dict(endpoints=len(endpoints),path_coverage=len(order)/len(coordinates),closed=len(endpoints)==0)\n\n\ndef _width(binary,path):\n    radius=distance_transform_edt(binary)\n    samples=map_coordinates(radius,[path[:,1],path[:,0]],order=1,mode=\'nearest\')\n    middle=samples[len(samples)//10:len(samples)-len(samples)//10]\n    return max(1.,float(np.median(2*middle-1)))\n\n\ndef render_reference(reference,original_path,target_path,width_scale):\n    """Render the deformed original centerline with the estimated stroke width.\n\n    Raster thin hints must not be stretched directly: their one-pixel staircase\n    and interpolation profile create serrated, gray thick strokes.\n    """\n    reference=_unit(reference);h,w=reference.shape\n    # Dense samples reduce nearest-segment seams while preserving reference caps.\n    # Keep normalized arc correspondence even when relative segment lengths change.\n    target=np.stack([np.interp(np.linspace(0,1,512),np.linspace(0,1,len(target_path)),target_path[:,i]) for i in range(2)],1)\n    yy,xx=np.mgrid[:h,:w];pixels=np.stack((xx.ravel(),yy.ravel()),1)\n    binary=reference>=max(.1,float(reference.max())*.3)\n    width=_width(binary,original_path)*width_scale\n    distance,_=cKDTree(target).query(pixels)\n    return np.clip(.5+(width/2-distance),0,1).reshape(h,w).astype(np.float32)\n\n\ndef _affine_fallback(reference,observed,config):\n    """Preserve a loop/branch prototype rather than cutting it into a path."""\n    def moments(mask):\n        y,x=np.nonzero(mask);xy=np.stack((x,y),1).astype(float)\n        center=xy.mean(0);extent=np.percentile(xy,95,axis=0)-np.percentile(xy,5,axis=0)\n        return center,np.maximum(extent,1)\n    source=reference>max(.1,float(reference.max())*.3)\n    rc,re=moments(source);tc,te=moments(observed)\n    shift=tc-rc;shift*=min(1.,config.max_displacement/max(np.linalg.norm(shift),1e-6))\n    scale=np.clip(te/re,.75,1.35)\n    # scipy uses y,x and maps output to input.\n    inverse=np.diag(1/scale[::-1]);offset=rc[::-1]-inverse@(rc+shift)[::-1]\n    result=affine_transform(reference,inverse,offset=offset,output_shape=reference.shape,order=1,mode=\'constant\')\n    return result.clip(0,1).astype(np.float32),dict(translation=shift.tolist(),scale=scale.tolist())\n\n\ndef fit_reference_stroke(reference,extracted,*,hint=None,config=None):\n    config=config or ReferenceFitConfig();reference=_unit(reference);extracted=_unit(extracted)\n    if reference.shape!=extracted.shape:raise ValueError(\'Reference/extraction shape mismatch\')\n    hint=reference if hint is None else _unit(hint)\n    if hint.shape!=reference.shape:raise ValueError(\'Hint shape mismatch\')\n    observed,component_info=_component(extracted,hint)\n    canonical,_=_component(reference)\n    original,path_info=principal_path(canonical,config.points)\n    metadata=dict(component_info,reference_path=path_info,warnings=[])\n    if path_info[\'closed\'] or path_info[\'path_coverage\']<.65 or (path_info[\'endpoints\']>2 and path_info[\'path_coverage\']<.9):\n        warped,transform=_affine_fallback(reference,observed,config)\n        metadata.update(method=\'affine_topology_preserving_fallback\',transform=transform)\n        metadata[\'warnings\'].append(\'Closed/branched prototype: affine fallback, no open-curve fit\')\n        return dict(warped=warped,hint=hint,reference=reference,extracted=extracted,\n            reference_path=None,target_path=None,hint_path=None,width_scale=1.,metadata=metadata)\n    try:\n        hint_binary,_=_component(hint)\n        initial,_=principal_path(hint_binary,config.points)\n        if np.linalg.norm(initial[[0,-1]]-original[[0,-1]])>np.linalg.norm(initial[::-1][[0,-1]]-original[[0,-1]]):initial=initial[::-1]\n        if np.max(np.linalg.norm(initial-original,axis=1))>config.max_displacement:\n            initial=original.copy();metadata[\'warnings\'].append(\'Learned hint displacement too large; initialized from reference\')\n    except ValueError:\n        initial=original.copy();metadata[\'warnings\'].append(\'No usable learned hint; initialized from reference\')\n    target,target_info=principal_path(observed,config.points)\n    if np.linalg.norm(target[[0,-1]]-initial[[0,-1]])>np.linalg.norm(target[::-1][[0,-1]]-initial[[0,-1]]):target=target[::-1]\n    metadata[\'extracted_path\']=target_info\n    if target_info[\'closed\']:\n        target=initial.copy();metadata[\'warnings\'].append(\'Closed extracted path for open reference: retained initialization\')\n    if target_info[\'path_coverage\']<.75:metadata[\'warnings\'].append(\'Side branches excluded by principal-path fit\')\n    if component_info[\'selected_mass_fraction\']<.65:metadata[\'warnings\'].append(\'Large disconnected extraction mass excluded\')\n    chord=original[-1]-original[0];length=np.linalg.norm(np.diff(original,axis=0),axis=1).sum()\n    normal=np.array([-chord[1],chord[0]])/max(np.linalg.norm(chord),1e-6)\n    deviation=np.max(np.abs((original-original[0])@normal))\n    straight=deviation<=1.1 or deviation/max(length,1)<.025\n    if straight:\n        # Robust TLS: trim off-axis centerline points before estimating direction.\n        points=target.copy()\n        for _ in range(3):\n            center=points.mean(0);_,_,v=np.linalg.svd(points-center,full_matrices=False);axis=v[0]\n            residual=np.abs((target-center)@np.array([-axis[1],axis[0]]))\n            points=target[residual<=max(.75,float(np.quantile(residual,.85)))]\n        projection=(target-center)@axis\n        if axis@(initial[-1]-initial[0])<0:axis=-axis;projection=-projection\n        endpoints=np.quantile(projection,[.01,.99])\n        target=center+np.linspace(*endpoints,config.points)[:,None]*axis\n        metadata[\'method\']=\'robust_straight_reference_fit\'\n    else:metadata[\'method\']=\'regularized_curved_reference_fit\'\n    target_length=np.linalg.norm(np.diff(target,axis=0),axis=1).sum()\n    ratio=target_length/max(length,1e-6)\n    if not config.min_length_ratio<=ratio<=config.max_length_ratio:\n        metadata[\'warnings\'].append(\'Extracted path length outside allowed range; retained bounded initialization\')\n        target=initial.copy()\n        if straight:target=np.linspace(target[0],target[-1],config.points)\n    n=config.points;second=np.diff(np.eye(n),n=2,axis=0)\n    system=(1+config.shape_prior)*np.eye(n)+config.smoothness*(second.T@second)\n    if straight:\n        fitted=target\n    else:\n        # Smooth the deformation, rather than tracing every observed bend/spur.\n        fitted=initial+np.linalg.solve(system,target-initial)\n        endpoints=fitted[[0,-1]].copy()\n        fitted=gaussian_filter1d(fitted,.8,axis=0,mode=\'nearest\')\n        fitted[0],fitted[-1]=endpoints\n    displacement=fitted-original\n    norm=np.linalg.norm(displacement,axis=1,keepdims=True)\n    limit=min(1.,config.max_displacement/max(float(norm.max()),1e-6)) if straight else np.minimum(1,config.max_displacement/np.maximum(norm,1e-6))\n    fitted=original+displacement*limit\n    ref_width=_width(canonical,original)\n    target_width=float(np.clip(_width(observed,target),config.min_width,config.max_width))\n    width_scale=float(np.clip(target_width/ref_width,.5,8.))\n    warped=render_reference(reference,original,fitted,width_scale)\n    metadata.update(width_reference=ref_width,width_observed=target_width,width_scale=width_scale,\n        extracted_length_ratio=float(ratio),max_displacement=float(np.linalg.norm(fitted-original,axis=1).max()))\n    return dict(warped=warped,hint=hint,reference=reference,extracted=extracted,\n        reference_path=original,target_path=fitted,hint_path=initial,width_scale=width_scale,metadata=metadata)\n\n\ndef fit_reference_glyph(strokes,extracted,*,hints=None,glyph_reference=None,config=None):\n    config=config or ReferenceFitConfig()\n    strokes=np.asarray(strokes,np.float32);extracted=np.asarray(extracted,np.float32)\n    if strokes.shape!=extracted.shape or strokes.ndim!=3:raise ValueError(\'K matching reference/extracted masks required\')\n    hints=strokes if hints is None else np.asarray(hints,np.float32)\n    if hints.shape!=strokes.shape:raise ValueError(\'Hint shape mismatch\')\n    fitted=[]\n    for k in range(len(strokes)):\n        try:item=fit_reference_stroke(strokes[k],extracted[k],hint=hints[k],config=config)\n        except ValueError as exc:\n            item=dict(warped=strokes[k].copy(),hint=hints[k],reference=strokes[k],extracted=extracted[k],\n                reference_path=None,target_path=None,hint_path=None,width_scale=1.,\n                metadata=dict(method=\'unreliable_reference_fallback\',warnings=[str(exc)]))\n        fitted.append(item)\n    # Joint bounded translations keep reference contacts from independently drifting.\n    matrix=np.eye(len(strokes));rhs=np.zeros((len(strokes),2));contacts=[]\n    for i,a in enumerate(fitted):\n        if a[\'reference_path\'] is None:continue\n        for j in range(i+1,len(fitted)):\n            b=fitted[j]\n            if b[\'reference_path\'] is None:continue\n            distance=np.linalg.norm(a[\'reference_path\'][:,None]-b[\'reference_path\'][None],axis=2)\n            u,v=np.unravel_index(distance.argmin(),distance.shape)\n            connected=distance[u,v]<=config.junction_radius\n            if not connected and glyph_reference is not None and distance[u,v]<=10:\n                line=np.linspace(a[\'reference_path\'][u],b[\'reference_path\'][v],32)\n                ink=map_coordinates(_unit(np.asarray(glyph_reference).squeeze()),[line[:,1],line[:,0]],order=1,mode=\'constant\')\n                connected=float((ink>.25).mean())>.85\n            if not connected:continue\n            # Sparse annotations often omit the segment inside a junction. If\n            # the full reference confirms a connection, extend an endpoint to\n            # the other stroke instead of translating its whole visible tip.\n            if glyph_reference is not None:\n                ia=0 if u<4 else -1 if u>=len(a[\'reference_path\'])-4 else None\n                ib=0 if v<4 else -1 if v>=len(b[\'reference_path\'])-4 else None\n                if (ia is None)!=(ib is None):\n                    end_item,end_index,other,other_index=(a,ia,b,v) if ia is not None else (b,ib,a,u)\n                    difference=other[\'target_path\'][other_index]-end_item[\'target_path\'][end_index]\n                    gap=float(np.linalg.norm(difference))\n                    if gap<=config.max_displacement:\n                        t=np.linspace(0,1,len(end_item[\'target_path\']))\n                        factor=(1-t)**3 if end_index==0 else t**3\n                        if end_item[\'metadata\'].get(\'method\')==\'robust_straight_reference_fit\':factor=(1-t) if end_index==0 else t\n                        end_item[\'target_path\']=end_item[\'target_path\']+factor[:,None]*difference\n                        end_item[\'metadata\'].setdefault(\'endpoint_extensions\',[]).append(gap)\n            difference=a[\'target_path\'][u]-b[\'target_path\'][v]\n            weight=config.junction_weight\n            matrix[i,i]+=weight;matrix[j,j]+=weight;matrix[i,j]-=weight;matrix[j,i]-=weight\n            rhs[i]-=weight*difference;rhs[j]+=weight*difference\n            contacts.append(dict(strokes=[i,j],indices=[int(u),int(v)],gap_before=float(np.linalg.norm(difference))))\n    shift=np.linalg.solve(matrix,rhs)\n    shift*=np.minimum(1,config.max_junction_shift/np.maximum(np.linalg.norm(shift,axis=1,keepdims=True),1e-6))\n    for i,item in enumerate(fitted):\n        if item[\'target_path\'] is None:continue\n        item[\'target_path\']=item[\'target_path\']+shift[i]\n        displacement=item[\'target_path\']-item[\'reference_path\']\n        maximum=float(np.linalg.norm(displacement,axis=1).max())\n        if maximum>config.max_displacement:\n            item[\'target_path\']=item[\'reference_path\']+displacement*(config.max_displacement/maximum)\n            item[\'metadata\'][\'warnings\'].append(\'Joint displacement clipped; inspect contact consistency\')\n        item[\'warped\']=render_reference(item[\'reference\'],item[\'reference_path\'],item[\'target_path\'],item[\'width_scale\'])\n        item[\'metadata\'][\'junction_translation\']=shift[i].tolist()\n        item[\'metadata\'][\'final_max_displacement\']=float(np.linalg.norm(item[\'target_path\']-item[\'reference_path\'],axis=1).max())\n    for contact in contacts:\n        i,j=contact[\'strokes\'];u,v=contact[\'indices\']\n        contact[\'gap_after\']=float(np.linalg.norm(fitted[i][\'target_path\'][u]-fitted[j][\'target_path\'][v]))\n    masks=np.stack([item[\'warped\'] for item in fitted])\n    return dict(strokes=masks,candidate=masks.max(0),fits=fitted,contacts=contacts,config=asdict(config))\n'}
ROOT=Path('/content/hsqm_adaptive_runtime');ROOT.mkdir(exist_ok=True)
for package in ('diffusion_model','stroke_model'):
    (ROOT/package).mkdir(exist_ok=True);(ROOT/package/'__init__.py').touch()
for name,code in RUNTIME_HELPERS.items():
    (ROOT/name).write_text(code,encoding='utf-8')
sys.path.insert(0,str(ROOT))
# Cell 1 can be rerun after a different notebook in the same runtime.
for name in list(sys.modules):
    if name.split('.')[0] in ('diffusion_model','stroke_model'):sys.modules.pop(name)
import torch,numpy as np,matplotlib.pyplot as plt
from PIL import Image
from google.colab import files
from diffusion_model.fitted_candidate_diffusion import load_case,image_tensor,run_pair
from stroke_model.adaptive_ink_constraints import load_stroke_bundle,checkpoint_hash,AdaptiveInkConstraints
from stroke_model.reference_model import ReferenceStrokeModel
if not torch.cuda.is_available():raise RuntimeError('런타임에서 L4 GPU를 선택하세요.')
print('GPU:',torch.cuda.get_device_name(0))
from stroke_model.guarded_ink_constraints import GuardedInkConstraints

In [ ]:
USE_DRIVE=True
ZIP_PATH=''  # sweep ZIP 또는 stroke_bundle이 들어 있는 adaptive 결과 ZIP.
CHECKPOINT=''  # 알고 있다면 직접 경로 지정. 비워두면 Drive best.pt 검색.
CASE_ID='case_003_strength_0p65_seed_100'
drive_root=Path('/content/drive/MyDrive')
if USE_DRIVE and not drive_root.is_dir():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print('Drive 연결 불가. 아래 직접 업로드로 계속할 수 있습니다:',type(exc).__name__)
if not ZIP_PATH:
    names=('adaptive_ink_reverse_20261111_075257_135265.zip','sweep_20261111_062740_406072.zip')
    matches=[Path('/content')/name for name in names if (Path('/content')/name).is_file()]
    if not matches and USE_DRIVE and drive_root.is_dir():
        print('Drive에서 기존 결과 ZIP을 찾습니다...',flush=True)
        for name in names:
            matches=list(drive_root.rglob(name))
            if matches:break
    if matches:ZIP_PATH=str(matches[0]);print('ZIP:',ZIP_PATH)
if not ZIP_PATH:
    print('원래 sweep 또는 stroke_bundle이 들어 있는 adaptive 결과 ZIP을 업로드하세요.')
    uploaded=files.upload();names=[n for n in uploaded if n.lower().endswith('.zip')]
    if len(names)!=1:raise ValueError('실험 ZIP 하나를 선택하세요.')
    ZIP_PATH=names[0]
try:records,expected_sha=load_stroke_bundle(ZIP_PATH,CASE_ID)
except KeyError as exc:raise ValueError('원래 sweep ZIP의 source_conditions/warped_slot 자료가 필요합니다.') from exc
source,candidate,metadata=load_case(ZIP_PATH,CASE_ID)
metadata.update(checkpoint_sha256=expected_sha,reference_bundle_case=CASE_ID)
print('필요한 모델 SHA256:',expected_sha)
if not CHECKPOINT:
    candidates=list(Path('/content').glob('*.pt'))
    if USE_DRIVE and drive_root.is_dir():
        print('Drive best.pt를 검사합니다. 이름이 같아도 해시가 다르면 건너뜁니다.',flush=True)
        candidates.extend(drive_root.rglob('best.pt'))
    for path in candidates:
        try:actual=checkpoint_hash(path)
        except OSError:continue
        if actual==expected_sha:
            CHECKPOINT=str(path);break
        print('다른 체크포인트, 건너뜀:',path,flush=True)
if not CHECKPOINT:
    print('일치하는 획 추출 체크포인트(.pt)를 업로드하세요.')
    uploaded=files.upload();names=[n for n in uploaded if n.lower().endswith('.pt')]
    if len(names)!=1:raise ValueError('체크포인트 하나를 선택하세요.')
    CHECKPOINT=names[0]
if checkpoint_hash(CHECKPOINT)!=expected_sha:
    raise ValueError('체크포인트 해시 불일치: 원래 sweep 실험에 사용한 모델이 아닙니다.')
state=torch.load(CHECKPOINT,map_location='cpu',weights_only=True)
if state.get('architecture')!='reference_v2_topology_loss_v1':
    raise ValueError('이 노트북은 기존 reference_v2_topology_loss_v1 체크포인트를 사용합니다.')
extractor=ReferenceStrokeModel(image_size=state['image_size'],width=state['width'],pretrained=False)
extractor.load_state_dict(state['model'],strict=True)
extractor=extractor.to('cuda').eval().requires_grad_(False)
if any(r['strokes'].shape[-2:]!=(state['image_size'],state['image_size']) for r in records):
    raise ValueError('기준 획과 모델 해상도가 다릅니다.')
del state
print('검증된 모델:',CHECKPOINT)

In [ ]:
ENABLE_RESTORE=False
REFRESH_EVERY=3
source_rgb=image_tensor(source,'cuda')
objective=GuardedInkConstraints(source_rgb,image_tensor(candidate,'cuda'),extractor,records,refresh_every=REFRESH_EVERY,enable_restore=ENABLE_RESTORE)
print('획별 허용 여유 (모델 좌표와 원본 이미지 좌표):')
for row in objective.bands:print(row)
erase=objective.erase[0,0].cpu().numpy();restore=objective.restore[0,0].cpu().numpy()
rgb=np.asarray(source,dtype=np.float32)/255
alpha=.7*np.maximum(erase,restore)[...,None]
color=np.stack([erase,np.zeros_like(erase),restore],-1)
color/=np.maximum(color.max(-1,keepdims=True),1e-8)
overlay=rgb*(1-alpha)+color*alpha
fig,axes=plt.subplots(1,4,figsize=(20,5))
for ax,im,title in zip(axes,[source,overlay,erase,restore],
        ['Original input','Red=erase / blue=restore','Initial erase pressure','Restore centerlines']):
    ax.imshow(im,cmap='magma',vmin=0,vmax=1);ax.set_title(title);ax.axis('off')
plt.show();plt.close(fig)
print('기본 삭제만 먼저 적용하며 새 제안은 반복 확인과 정상 획 보호 검사를 통과해야 합니다.')

In [ ]:
from datetime import datetime
from huggingface_hub import model_info
import shutil
STEPS=60
STRENGTH=.25  # 실제 추가 역확산 15단계. 이전 버전과 동일.
SEED=2026
INNER_STEPS=4
STEP_RATIO=.012
MODEL_ID='stable-diffusion-v1-5/stable-diffusion-v1-5'
REVISION=model_info(MODEL_ID).sha
OUTPUT=Path('/content')/('guarded_ink_release_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
# Cell 4 reruns always reset ownership/mask history to the same original input.
objective=GuardedInkConstraints(source_rgb,image_tensor(candidate,'cuda'),extractor,records,refresh_every=REFRESH_EVERY,enable_restore=ENABLE_RESTORE)
def progress(method,step,total,row):
    if step==1 or step%3==0 or step==total or row.get('constraint_refresh'):
        print(f"{method} {step}/{total} | refresh={bool(row.get('constraint_refresh'))} | "
              f"accepted={row['accepted_inner_steps']} | guard_rejected={row.get('rejected_guard_attempts',0)} | {row.get('loss_before')} → {row.get('loss_after')}",flush=True)
report=run_pair(source,candidate,metadata,OUTPUT,steps=STEPS,strength=STRENGTH,seed=SEED,
    inner_steps=INNER_STEPS,step_ratio=STEP_RATIO,model_id=MODEL_ID,revision=REVISION,
    progress=progress,guidance_mode='adaptive_ink',objective_override=objective)
panels=[('Original input',source),('Re-denoise only',Image.open(OUTPUT/'redenoise_only.png')),
        ('Raw guarded reverse',Image.open(OUTPUT/'adaptive_guided.png')),('Selected output',Image.open(OUTPUT/'selected.png'))]
fig,axes=plt.subplots(1,4,figsize=(24,6))
for ax,(title,im) in zip(axes,panels):ax.imshow(im);ax.set_title(title);ax.axis('off')
fig.tight_layout();fig.savefig(OUTPUT/'comparison.png',dpi=140);plt.show();plt.close(fig)
print('최종 선택:',json.dumps(report['selection'],ensure_ascii=False,indent=2))
print('공통 초기 마스크에서 평가한 진단값 (정확도 아님):',json.dumps(report['metrics'],indent=2))
print('재추출 이력:',[(r['step'],r['erase_pixels']) for r in report['constraint_history']])
mask_paths=sorted(OUTPUT.glob('erase_step_*.png'))
if mask_paths:
    fig,axes=plt.subplots(1,len(mask_paths),figsize=(4*len(mask_paths),4),squeeze=False)
    for ax,path in zip(axes.flat,mask_paths):
        ax.imshow(source);ax.imshow(Image.open(path),cmap='Reds',vmin=0,vmax=255,alpha=.5)
        ax.set_title(path.stem);ax.axis('off')
    fig.tight_layout();fig.savefig(OUTPUT/'refresh_masks.png',dpi=120);plt.show();plt.close(fig)
paths=sorted(OUTPUT.glob('adaptive_guided_step_*.png'))
fig,axes=plt.subplots(1,len(paths),figsize=(4*len(paths),4),squeeze=False)
for ax,path in zip(axes.flat,paths):
    ax.imshow(Image.open(path));ax.set_title(path.stem);ax.axis('off')
fig.tight_layout();fig.savefig(OUTPUT/'adaptive_trajectory.png',dpi=120);plt.show();plt.close(fig)
(OUTPUT/'runtime_helpers.json').write_text(json.dumps(RUNTIME_HELPERS,ensure_ascii=False),encoding='utf-8')
# Preserve the small stroke bundle for subsequent experiments, not the checkpoint.
with zipfile.ZipFile(ZIP_PATH) as z:
    from pathlib import PurePosixPath
    for name in z.namelist():
        member=PurePosixPath(name)
        if member.is_absolute() or '..' in member.parts or '\\' in name:continue
        clean=name.removeprefix('stroke_bundle/')
        if clean.startswith('source_conditions/') or clean.startswith(CASE_ID+'/warped_slot_'):
            path=OUTPUT/'stroke_bundle'/clean;path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(z.read(name))
archive=shutil.make_archive(str(OUTPUT),'zip',OUTPUT)
print('실제 GPU 결과와 정상 획 변형을 함께 확인하세요:',archive)
files.download(archive)